# Website FAQ Chatbot — From URL to RAG

## 1. Project Overview

**Task:** Crawl a website, extract its text content, build a vector store, and create a grounded FAQ chatbot that answers questions using only the website's content.

**Approach:** Website-to-RAG pipeline — fetch pages → clean HTML → chunk text → embed → retrieve → generate grounded answers with citations.

**Stack:**
- `requests` + `BeautifulSoup` — web crawling and HTML parsing
- `LangChain` + `RecursiveCharacterTextSplitter` — text chunking
- `HuggingFaceEmbeddings` (`all-MiniLM-L6-v2`) — local embeddings
- `ChromaDB` — vector store for retrieval
- `ChatOllama` + `qwen3.5:9b` — local answer generation

> **No API keys required.** Everything runs locally.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | **Crawl** a website and follow internal links to collect pages |
| 2 | **Extract** clean text from HTML (strip nav, footer, scripts) |
| 3 | **Chunk** extracted text with overlap to preserve context |
| 4 | **Embed** chunks using a local sentence-transformer model |
| 5 | **Store** embeddings in ChromaDB for similarity search |
| 6 | **Retrieve** relevant chunks given a user question |
| 7 | **Generate** grounded answers that cite their sources |
| 8 | **Evaluate** answer quality: faithfulness and relevance |

## 3. Problem Statement

Every organization has a website full of useful information — product docs, FAQs, policies, pricing — but users struggle to find answers. A FAQ chatbot lets users ask natural-language questions and get direct answers grounded in the website content.

The challenge: websites are HTML, not plain text. You need to:
1. **Crawl** pages (follow links, handle navigation)
2. **Clean** HTML (remove boilerplate: nav bars, footers, ads, scripts)
3. **Chunk** intelligently (not mid-sentence, with enough context)
4. **Ground** answers (never hallucinate — only answer from retrieved content)

## 4. Why This Project Matters

- **RAG over websites** is the #1 enterprise LLM use case (support bots, internal knowledge bases)
- **HTML cleaning** is a critical but under-taught skill — garbage in, garbage out
- **Grounded generation** (answering ONLY from context) prevents hallucination
- Every company with a docs site wants this: Zendesk, Intercom, and Drift all offer AI FAQ features

## 5. Pipeline Architecture

```
Website URL(s)
      │
      ▼
  [Crawl] ── fetch HTML, follow internal links, respect robots.txt
      │
      ▼
  [Extract] ── strip HTML tags, remove boilerplate (nav, footer, scripts)
      │
      ▼
  [Chunk] ── RecursiveCharacterTextSplitter (500 chars, 100 overlap)
      │
      ▼
  [Embed] ── all-MiniLM-L6-v2 sentence embeddings (384 dims)
      │
      ▼
  [Store] ── ChromaDB vector store with source metadata
      │
      ▼
  [Query] ── user question → embed → top-k retrieval
      │
      ▼
  [Generate] ── LLM answers from retrieved context only
      │
      ▼
  [Cite] ── include source URLs in the answer
```

## 6. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core langchain-huggingface
# !pip install -q langchain-text-splitters chromadb beautifulsoup4 requests

print("Dependencies: langchain, chromadb, beautifulsoup4, requests, sentence-transformers")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 7. Imports

In [ ]:
import os
import re
import json
import time
import warnings
from urllib.parse import urljoin, urlparse

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

import requests
from bs4 import BeautifulSoup

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

print("All imports OK")

## 8. Configuration

In [ ]:
LLM_MODEL = "qwen3.5:9b"
EMBED_MODEL = "all-MiniLM-L6-v2"

# ── Chunking parameters ───────────────────────────────
CHUNK_SIZE = 500       # Characters per chunk
CHUNK_OVERLAP = 100    # Overlap between chunks

# ── Retrieval parameters ──────────────────────────────
TOP_K = 4              # Number of chunks to retrieve per query

# ── Crawler settings ──────────────────────────────────
MAX_PAGES = 15         # Maximum pages to crawl
REQUEST_TIMEOUT = 10   # Seconds before a request times out

print("Configuration loaded")
print(f"  LLM: {LLM_MODEL}")
print(f"  Embeddings: {EMBED_MODEL}")
print(f"  Chunk: {CHUNK_SIZE} chars, {CHUNK_OVERLAP} overlap")
print(f"  Retrieval: top-{TOP_K}")

---

# Part A — Website Crawling & Text Extraction

## 9. Build the Web Crawler

The crawler starts from a seed URL, fetches the page, extracts links, and follows internal links (same domain) up to a configurable depth. It respects basic politeness by not re-fetching visited URLs.

### Key design choices
- **Internal links only** — we stay on the same domain
- **Skip non-HTML** — ignore PDFs, images, CSS, JS files
- **Dedup by URL** — don't fetch the same page twice
- **Timeout protection** — don't hang on slow pages

In [ ]:
SKIP_EXTENSIONS = {
    ".pdf", ".png", ".jpg", ".jpeg", ".gif", ".svg", ".webp",
    ".css", ".js", ".ico", ".xml", ".zip", ".tar", ".gz",
    ".mp3", ".mp4", ".avi", ".mov", ".woff", ".woff2", ".ttf",
}


def should_skip_url(url: str) -> bool:
    """Skip non-HTML resources and fragments."""
    parsed = urlparse(url)
    path = parsed.path.lower()
    return any(path.endswith(ext) for ext in SKIP_EXTENSIONS)


def extract_text(html: str, url: str) -> dict:
    """
    Extract clean text from HTML, removing boilerplate.

    Strategy:
    1. Remove <script>, <style>, <nav>, <footer>, <header> tags entirely
    2. Extract text from <main>, <article>, or <body> (in that priority)
    3. Clean up whitespace
    """
    soup = BeautifulSoup(html, "html.parser")

    # Get title
    title = soup.title.string.strip() if soup.title and soup.title.string else ""

    # Remove boilerplate elements
    for tag in soup.find_all(["script", "style", "nav", "footer", "header",
                              "aside", "noscript", "iframe"]):
        tag.decompose()

    # Prefer main content areas
    content_area = (
        soup.find("main")
        or soup.find("article")
        or soup.find("div", {"role": "main"})
        or soup.find("div", class_=re.compile(r"content|main|body", re.I))
        or soup.body
        or soup
    )

    # Extract text
    text = content_area.get_text(separator="\n", strip=True)

    # Clean up: collapse blank lines, strip excessive whitespace
    lines = [line.strip() for line in text.split("\n") if line.strip()]
    clean_text = "\n".join(lines)

    return {"url": url, "title": title, "text": clean_text}


def extract_links(html: str, base_url: str, base_domain: str) -> list[str]:
    """Extract internal links from a page."""
    soup = BeautifulSoup(html, "html.parser")
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"].split("#")[0].strip()  # Remove fragments
        if not href or href.startswith(("mailto:", "tel:", "javascript:")):
            continue
        full_url = urljoin(base_url, href)
        if urlparse(full_url).netloc == base_domain and not should_skip_url(full_url):
            links.append(full_url)
    return list(set(links))


def crawl_website(seed_url: str, max_pages: int = MAX_PAGES) -> list[dict]:
    """
    Crawl a website starting from seed_url.
    Returns list of {url, title, text} dicts.
    """
    base_domain = urlparse(seed_url).netloc
    visited = set()
    queue = [seed_url]
    pages = []

    headers = {
        "User-Agent": "Mozilla/5.0 (Educational Bot) LearningProject/1.0"
    }

    while queue and len(pages) < max_pages:
        url = queue.pop(0)
        if url in visited:
            continue
        visited.add(url)

        try:
            resp = requests.get(url, headers=headers, timeout=REQUEST_TIMEOUT)
            if resp.status_code != 200:
                continue
            content_type = resp.headers.get("Content-Type", "")
            if "text/html" not in content_type:
                continue
        except (requests.RequestException, Exception) as e:
            print(f"  ⚠ Failed: {url[:60]} ({e})")
            continue

        # Extract text
        page = extract_text(resp.text, url)
        if len(page["text"]) < 50:  # Skip near-empty pages
            continue

        pages.append(page)
        print(f"  ✓ [{len(pages):2d}/{max_pages}] {page['title'][:50]:50s} ({len(page['text']):,} chars)")

        # Extract and queue internal links
        new_links = extract_links(resp.text, url, base_domain)
        queue.extend(link for link in new_links if link not in visited)

    return pages


print("Crawler functions defined")
print("  crawl_website() — BFS crawler with boilerplate removal")
print("  extract_text()  — HTML → clean text with title")
print("  extract_links() — internal link discovery")

## 10. Crawl a Website

We crawl **Python's official documentation** (`docs.python.org/3/faq/`) as our target. It's a great test case because:
- Public, well-structured pages
- FAQ format with distinct question-answer pairs
- Multiple linked sub-pages

If the live site is unreachable, we fall back to **sample content** so the notebook remains fully runnable offline.

In [ ]:
# ── Sample content (fallback if network is unavailable) ──
SAMPLE_PAGES = [
    {
        "url": "https://example.com/about",
        "title": "About TechCorp",
        "text": (
            "About TechCorp\n\n"
            "TechCorp is a leading provider of cloud-based software solutions for small and "
            "medium businesses. Founded in 2018, we serve over 10,000 customers across 45 countries.\n\n"
            "Our Mission\n"
            "We believe every business deserves enterprise-grade tools at affordable prices. "
            "Our platform combines project management, customer relationship management (CRM), "
            "and invoicing into one seamless experience.\n\n"
            "Our Team\n"
            "We have 150 employees distributed across 12 countries. Our engineering team of 80 "
            "developers builds and maintains the platform. We are headquartered in Austin, Texas "
            "with offices in London and Singapore."
        ),
    },
    {
        "url": "https://example.com/pricing",
        "title": "Pricing — TechCorp",
        "text": (
            "Pricing Plans\n\n"
            "Starter Plan — $9/month per user\n"
            "Includes: Project management, basic reporting, 5GB storage, email support.\n"
            "Best for: Teams of 1-5 people getting started.\n\n"
            "Professional Plan — $29/month per user\n"
            "Includes: Everything in Starter plus CRM, invoicing, custom fields, 50GB storage, "
            "priority support, and API access.\n"
            "Best for: Growing teams of 5-50 people.\n\n"
            "Enterprise Plan — Custom pricing\n"
            "Includes: Everything in Professional plus SSO (SAML), dedicated account manager, "
            "99.99% SLA, unlimited storage, custom integrations, and on-premise deployment option.\n"
            "Best for: Organizations with 50+ users or specific compliance needs.\n\n"
            "All plans include a 14-day free trial. No credit card required. "
            "Annual billing saves 20%. Educational and nonprofit discounts available — contact sales."
        ),
    },
    {
        "url": "https://example.com/faq",
        "title": "Frequently Asked Questions — TechCorp",
        "text": (
            "Frequently Asked Questions\n\n"
            "How do I reset my password?\n"
            "Click 'Forgot Password' on the login page. Enter your email address and we'll send "
            "a reset link within 2 minutes. The link expires after 24 hours. If you don't receive "
            "the email, check your spam folder or contact support@techcorp.com.\n\n"
            "Can I cancel my subscription at any time?\n"
            "Yes, you can cancel anytime from Settings > Billing > Cancel Subscription. "
            "Your account remains active until the end of your current billing period. "
            "We do not offer partial refunds for unused time. Annual plans can be cancelled "
            "but are non-refundable after 30 days.\n\n"
            "Do you offer an API?\n"
            "Yes, our REST API is available on Professional and Enterprise plans. "
            "Documentation is at api.techcorp.com/docs. Rate limits: 100 requests/minute on "
            "Professional, 1000 requests/minute on Enterprise. We also offer webhooks for "
            "real-time event notifications.\n\n"
            "What integrations do you support?\n"
            "We integrate with Slack, Microsoft Teams, Google Workspace, Salesforce, QuickBooks, "
            "Stripe, Zapier, and GitHub. Custom integrations are available via our API on "
            "Professional and Enterprise plans.\n\n"
            "Is my data secure?\n"
            "Yes. We use AES-256 encryption at rest and TLS 1.3 in transit. We are SOC 2 Type II "
            "certified and GDPR compliant. Data is stored in AWS data centers with 99.99% uptime. "
            "Enterprise customers can choose their data region (US, EU, or APAC).\n\n"
            "How do I contact support?\n"
            "Email support@techcorp.com for all plans (response within 24 hours). "
            "Professional plans get priority support (response within 4 hours). "
            "Enterprise plans include a dedicated account manager and phone support. "
            "Our support hours are Monday-Friday, 9am-6pm EST."
        ),
    },
    {
        "url": "https://example.com/features",
        "title": "Features — TechCorp",
        "text": (
            "Platform Features\n\n"
            "Project Management\n"
            "Create projects with Kanban boards, Gantt charts, and list views. Assign tasks to "
            "team members with due dates, priorities, and custom labels. Track time spent on tasks "
            "with our built-in timer. Set up automated workflows to move tasks between stages.\n\n"
            "CRM (Customer Relationship Management)\n"
            "Manage contacts, companies, and deals in a visual pipeline. Track email conversations "
            "and meeting notes. Set reminders for follow-ups. "
            "Generate revenue forecasts based on deal stages and historical close rates. "
            "Available on Professional and Enterprise plans.\n\n"
            "Invoicing & Billing\n"
            "Create professional invoices with your company branding. Accept payments via "
            "Stripe and PayPal. Set up recurring invoices for subscription clients. "
            "Track outstanding payments and send automated reminders. "
            "Available on Professional and Enterprise plans.\n\n"
            "Reporting & Analytics\n"
            "Starter: Basic dashboards with project status and task completion rates.\n"
            "Professional: Custom reports, time tracking analytics, CRM pipeline reports, "
            "revenue dashboards.\n"
            "Enterprise: Advanced analytics with data export, custom KPIs, and scheduled "
            "report delivery via email.\n\n"
            "Mobile Apps\n"
            "Native iOS and Android apps with full offline support. Sync automatically "
            "when connected. Includes push notifications for task assignments and mentions."
        ),
    },
    {
        "url": "https://example.com/security",
        "title": "Security — TechCorp",
        "text": (
            "Security & Compliance\n\n"
            "Data Encryption\n"
            "All data is encrypted using AES-256 at rest and TLS 1.3 in transit. Database backups "
            "are encrypted and stored in geographically separate locations.\n\n"
            "Certifications\n"
            "SOC 2 Type II certified (audited annually). GDPR compliant with Data Processing "
            "Agreement available. HIPAA compliance available on Enterprise plans with BAA.\n\n"
            "Infrastructure\n"
            "Hosted on AWS with multi-AZ deployment. 99.99% uptime SLA for Enterprise customers. "
            "Automatic failover and disaster recovery with RPO < 1 hour and RTO < 4 hours.\n\n"
            "Access Controls\n"
            "Role-based access control (RBAC) with customizable permissions. "
            "SSO via SAML 2.0 (Enterprise plan). Two-factor authentication (2FA) available "
            "on all plans. Audit logs retained for 1 year (Enterprise: unlimited).\n\n"
            "Data Residency\n"
            "Enterprise customers can choose their data region: US (Virginia), EU (Frankfurt), "
            "or APAC (Singapore). Data never leaves the selected region."
        ),
    },
]

print(f"Sample content ready: {len(SAMPLE_PAGES)} pages")
for p in SAMPLE_PAGES:
    print(f"  {p['title']:40s} ({len(p['text']):,} chars)")

In [ ]:
# ── Try live crawl; fall back to samples if offline ───
TARGET_URL = "https://docs.python.org/3/faq/"

try:
    print(f"Attempting to crawl: {TARGET_URL}")
    print("-" * 60)
    pages = crawl_website(TARGET_URL, max_pages=10)
    if len(pages) < 2:
        raise ValueError("Too few pages fetched")
    print(f"\nCrawled {len(pages)} pages from {TARGET_URL}")
    data_source = "live"
except Exception as e:
    print(f"\n⚠ Live crawl failed ({e}). Using sample content instead.")
    pages = SAMPLE_PAGES
    data_source = "sample"

# Summary
total_chars = sum(len(p["text"]) for p in pages)
print(f"\nData source: {data_source}")
print(f"Total pages: {len(pages)}")
print(f"Total text:  {total_chars:,} characters")

## 11. Inspect Extracted Content

Before chunking, verify the extraction quality — are we getting clean text without HTML artifacts?

In [ ]:
for i, page in enumerate(pages):
    print(f"\nPage {i+1}: {page['title']}")
    print(f"URL: {page['url']}")
    print(f"Text length: {len(page['text']):,} chars")
    print(f"Preview: {page['text'][:200]}...")
    print("-" * 60)

---

# Part B — Chunking & Embedding

## 12. Text Chunking

### Why chunk?
- Embedding models work best on **paragraph-sized** text (not full pages)
- Retrieval is more precise with **smaller, focused** chunks
- Full pages may contain multiple topics, diluting the embedding

### Chunking strategy
- `RecursiveCharacterTextSplitter` tries to split at paragraph breaks, then sentences, then words
- **Overlap** ensures no information is lost at chunk boundaries
- Each chunk keeps its **source URL** as metadata for citation

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len,
)

# Convert pages to LangChain Documents with metadata
documents = []
for page in pages:
    chunks = splitter.split_text(page["text"])
    for j, chunk in enumerate(chunks):
        doc = Document(
            page_content=chunk,
            metadata={
                "source": page["url"],
                "title": page["title"],
                "chunk_index": j,
            },
        )
        documents.append(doc)

print(f"Chunking complete")
print(f"  Total documents: {len(documents)}")
chunk_lens = [len(d.page_content) for d in documents]
print(f"  Chunk size — min: {min(chunk_lens)}, max: {max(chunk_lens)}, "
      f"avg: {sum(chunk_lens)/len(chunk_lens):.0f} chars")

# Show a few sample chunks
print(f"\nSample chunks:")
for doc in documents[:3]:
    print(f"  [{doc.metadata['title'][:30]:30s} | chunk {doc.metadata['chunk_index']}]")
    print(f"  {doc.page_content[:120]}...\n")

## 13. Create Embeddings & Vector Store

We use `all-MiniLM-L6-v2` — a fast, 384-dimensional sentence embedding model that runs locally. ChromaDB stores the vectors in-memory for this demo.

In [ ]:
# ── Initialize embedding model ────────────────────────
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cpu"},
)

# Quick test
test_embedding = embeddings.embed_query("test embedding")
print(f"Embedding model loaded: {EMBED_MODEL}")
print(f"  Dimensions: {len(test_embedding)}")

In [ ]:
# ── Build vector store ────────────────────────────────
t0 = time.perf_counter()

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="website_faq",
)

elapsed = time.perf_counter() - t0
print(f"Vector store created in {elapsed:.1f}s")
print(f"  Documents indexed: {vectorstore._collection.count()}")

## 14. Test Retrieval

Before connecting the LLM, verify that the retriever returns relevant chunks for sample questions.

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

test_queries = [
    "What is the pricing?",
    "How do I reset my password?",
    "Is my data encrypted?",
    "What integrations are available?",
]

print("RETRIEVAL TESTS")
print("=" * 60)

for query in test_queries:
    docs = retriever.invoke(query)
    print(f"\nQ: {query}")
    for j, doc in enumerate(docs):
        print(f"  [{j+1}] ({doc.metadata['title'][:30]}) {doc.page_content[:100]}...")

---

# Part C — Grounded Answer Generation

## 15. The RAG Answer Prompt

### Grounding rules

The most important aspect of a FAQ chatbot: **never hallucinate**. The prompt must enforce:
1. Answer ONLY from the retrieved context
2. If the context doesn't contain the answer, say so explicitly
3. Cite which source the information came from

In [ ]:
llm = ChatOllama(model=LLM_MODEL, temperature=0)

# Quick test
test = llm.invoke([HumanMessage(content="Say ready.")])
raw = test.content
if "<think>" in raw:
    raw = raw.split("</think>")[-1].strip()
print(f"LLM ready: {raw[:30]}")

In [ ]:
RAG_SYSTEM = """You are a helpful FAQ assistant for a website. Answer the user's question 
using ONLY the provided context. 

STRICT RULES:
1. Only use information from the CONTEXT below. Do not use prior knowledge.
2. If the context does not contain enough information to answer, say: 
   "I don't have enough information from the website to answer this question."
3. Cite your sources by mentioning the page title in parentheses.
4. Be concise — answer in 2-4 sentences unless more detail is needed.
5. If the user asks about something not covered by the website, do NOT guess."""

RAG_USER = """CONTEXT:
{context}

QUESTION: {question}

Answer (with citations):"""


def clean(text: str) -> str:
    """Strip thinking tags."""
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def format_context(docs: list) -> str:
    """Format retrieved documents as numbered context blocks."""
    blocks = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get("title", doc.metadata.get("source", "unknown"))
        blocks.append(f"[Source {i+1}: {source}]\n{doc.page_content}")
    return "\n\n".join(blocks)


def ask_faq(question: str, verbose: bool = True) -> dict:
    """
    Full RAG pipeline: retrieve → format context → generate answer.
    """
    t0 = time.perf_counter()

    # Retrieve
    docs = retriever.invoke(question)
    context = format_context(docs)

    # Generate
    resp = llm.invoke([
        SystemMessage(content=RAG_SYSTEM),
        HumanMessage(content=RAG_USER.format(context=context, question=question)),
    ])
    answer = clean(resp.content)
    elapsed = time.perf_counter() - t0

    sources = list({doc.metadata.get("title", doc.metadata["source"]) for doc in docs})

    result = {
        "question": question,
        "answer": answer,
        "sources": sources,
        "num_chunks": len(docs),
        "time_sec": round(elapsed, 1),
    }

    if verbose:
        print(f"Q: {question}")
        print(f"A: {answer}")
        print(f"Sources: {', '.join(sources)}")
        print(f"Time: {elapsed:.1f}s\n")

    return result


print("FAQ pipeline defined: ask_faq(question)")

## 16. Ask Questions

Test the FAQ chatbot with a range of questions — some answerable from the website, some not.

In [ ]:
FAQ_QUESTIONS = [
    # ── Questions answerable from the content ─────────
    "What plans do you offer and what do they cost?",
    "How do I reset my password?",
    "Is my data encrypted? What certifications do you have?",
    "Do you have an API? What are the rate limits?",
    "What integrations does TechCorp support?",
    "Can I cancel my subscription?",

    # ── Questions NOT answerable from the content ─────
    "What programming language is TechCorp built with?",
    "Who is the CEO of TechCorp?",
]

print("FAQ CHATBOT — QUESTION ANSWERING")
print("=" * 60)

all_results = []
for q in FAQ_QUESTIONS:
    result = ask_faq(q)
    all_results.append(result)
    print("-" * 60)

## 17. Grounding Check — Does the Chatbot Refuse to Hallucinate?

A critical test: when the answer isn't in the website, does the chatbot correctly refuse to answer?

In [ ]:
GROUNDING_TESTS = [
    # Out-of-scope questions that should get a refusal
    "What's the weather like today?",
    "Can you write Python code for me?",
    "What is the meaning of life?",
    "How does competitor XYZ compare to TechCorp?",
]

print("GROUNDING TEST — Expected: refusal / 'I don't have enough information'")
print("=" * 60)

refusal_keywords = ["don't have", "not enough", "cannot", "no information",
                    "not covered", "not mentioned", "unable", "doesn't contain"]

grounding_pass = 0
for q in GROUNDING_TESTS:
    result = ask_faq(q, verbose=False)
    answer_lower = result["answer"].lower()
    refused = any(kw in answer_lower for kw in refusal_keywords)
    icon = "✅" if refused else "❌"
    grounding_pass += int(refused)
    print(f"  {icon} Q: {q}")
    print(f"     A: {result['answer'][:150]}")
    print()

print(f"Grounding score: {grounding_pass}/{len(GROUNDING_TESTS)} correctly refused")

---

# Part D — Evaluation & Analysis

## 18. Answer Quality Evaluation — LLM-as-Judge

We use the LLM itself to score answers on two dimensions:
- **Faithfulness** — is the answer fully supported by the retrieved context?
- **Relevance** — does the answer actually address the question?

In [ ]:
JUDGE_SYSTEM = """You evaluate the quality of a FAQ chatbot's answers.
Score on two dimensions (1-5 each):

FAITHFULNESS (is the answer supported by the context?):
  5 = Fully supported, no unsupported claims
  3 = Mostly supported, minor additions
  1 = Contains significant hallucinations

RELEVANCE (does the answer address the question?):
  5 = Directly and completely answers the question
  3 = Partially relevant, misses key points
  1 = Doesn't address the question at all

Return ONLY JSON: {"faithfulness": <1-5>, "relevance": <1-5>, "explanation": "one sentence"}"""

JUDGE_USER = """Question: {question}
Answer: {answer}
Context used: {context}

Evaluate:"""


def parse_json_safe(text: str):
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    end = text.rfind("}") + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


# Evaluate the answerable questions
print("ANSWER QUALITY EVALUATION")
print("=" * 60)

eval_results = []
for result in all_results[:6]:  # Only the answerable ones
    # Get the context that was used
    docs = retriever.invoke(result["question"])
    context = format_context(docs)

    resp = llm.invoke([
        SystemMessage(content=JUDGE_SYSTEM),
        HumanMessage(content=JUDGE_USER.format(
            question=result["question"],
            answer=result["answer"],
            context=context[:1000],
        )),
    ])
    parsed = parse_json_safe(resp.content)
    if parsed:
        eval_results.append(parsed)
        f = parsed.get('faithfulness', 0)
        r = parsed.get('relevance', 0)
        print(f"  Q: {result['question'][:50]:50s} | Faith: {f}/5 | Relev: {r}/5")
    else:
        print(f"  Q: {result['question'][:50]:50s} | (parse error)")

if eval_results:
    avg_f = sum(e.get("faithfulness", 0) for e in eval_results) / len(eval_results)
    avg_r = sum(e.get("relevance", 0) for e in eval_results) / len(eval_results)
    print(f"\nAverage Faithfulness: {avg_f:.1f}/5")
    print(f"Average Relevance:   {avg_r:.1f}/5")

## 19. Retrieval Quality Analysis

Is the retriever finding the right chunks? If retrieval is bad, the answer will be bad regardless of the LLM.

In [ ]:
print("RETRIEVAL ANALYSIS")
print("=" * 60)

for result in all_results[:6]:
    docs = retriever.invoke(result["question"])

    # Use similarity search with scores
    scored = vectorstore.similarity_search_with_relevance_scores(result["question"], k=TOP_K)

    print(f"\nQ: {result['question']}")
    for doc, score in scored:
        print(f"  Score: {score:.3f} | [{doc.metadata['title'][:25]:25s}] {doc.page_content[:60]}...")

    # Check: are the top chunks from the most relevant page?
    sources = [doc.metadata["title"] for doc, _ in scored]
    unique = list(dict.fromkeys(sources))  # Deduplicate preserving order
    print(f"  Sources: {' → '.join(s[:20] for s in unique)}")

## 20. Naive vs RAG Comparison

Compare the RAG chatbot (grounded in website content) against the LLM answering without context.

In [ ]:
NAIVE_SYSTEM = """Answer the user's question. If you don't know, say so."""

compare_questions = [
    "What plans does TechCorp offer?",
    "Is TechCorp HIPAA compliant?",
    "How do I contact TechCorp support?",
]

print("NAIVE (no context) vs RAG (with website context)")
print("=" * 70)

for q in compare_questions:
    # Naive
    resp_naive = llm.invoke([
        SystemMessage(content=NAIVE_SYSTEM),
        HumanMessage(content=q),
    ])
    naive_answer = clean(resp_naive.content)

    # RAG
    rag_result = ask_faq(q, verbose=False)

    print(f"\nQ: {q}")
    print(f"  NAIVE: {naive_answer[:200]}")
    print(f"  RAG:   {rag_result['answer'][:200]}")
    print(f"  Sources: {', '.join(rag_result['sources'])}")
    print("-" * 70)

print("\n→ Notice: Naive answers are generic or hallucinated.")
print("→ RAG answers are specific and grounded in actual website content.")

---

## 21. Experiment — Chunk Size Impact

How does chunk size affect retrieval quality? Smaller chunks are more precise but may lack context. Larger chunks provide more context but may be less focused.

In [ ]:
CHUNK_SIZES = [200, 500, 1000]
test_question = "What security certifications does TechCorp have?"

print(f"CHUNK SIZE EXPERIMENT")
print(f"Question: {test_question}")
print("=" * 60)

for size in CHUNK_SIZES:
    # Re-chunk
    test_splitter = RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=size // 5,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    test_docs = []
    for page in pages:
        chunks = test_splitter.split_text(page["text"])
        for j, chunk in enumerate(chunks):
            test_docs.append(Document(
                page_content=chunk,
                metadata={"source": page["url"], "title": page["title"], "chunk_index": j},
            ))

    # Build temp vector store
    test_vs = Chroma.from_documents(test_docs, embeddings, collection_name=f"test_{size}")
    test_ret = test_vs.as_retriever(search_kwargs={"k": TOP_K})

    # Retrieve
    results = test_ret.invoke(test_question)
    avg_len = sum(len(d.page_content) for d in results) / len(results)

    print(f"\nChunk size: {size} chars (overlap: {size // 5})")
    print(f"  Total chunks: {len(test_docs)}")
    print(f"  Retrieved avg length: {avg_len:.0f} chars")
    for d in results[:2]:
        print(f"    {d.page_content[:120]}...")

    # Cleanup
    test_vs.delete_collection()

## 22. Edge Cases & Error Analysis

In [ ]:
EDGE_CASES = [
    # Ambiguous question matching multiple topics
    "Tell me about the Professional plan",

    # Very specific question requiring exact detail
    "What is the API rate limit on the Enterprise plan?",

    # Comparison question requiring info from multiple pages
    "What's the difference between Starter and Professional plans?",

    # Negation / tricky phrasing
    "What features are NOT included in the Starter plan?",
]

print("EDGE CASE ANALYSIS")
print("=" * 60)

for q in EDGE_CASES:
    result = ask_faq(q)
    print("-" * 60)

## 23. Common Mistakes

| Mistake | Why It's Wrong | What to Do Instead |
|---------|---------------|-------------------|
| **Not cleaning HTML** | Nav bars, footers, and cookie banners pollute embeddings | Strip `<script>`, `<style>`, `<nav>`, `<footer>` before extracting text |
| **Chunking too large** | Full-page chunks dilute the embedding — retrieval precision drops | Use 300-500 char chunks with overlap |
| **No grounding constraint** | LLM happily invents answers not in the website | System prompt must say "ONLY answer from context" |
| **No "I don't know" path** | Chatbot always gives an answer, even on off-topic questions | Explicitly instruct: "If context doesn't contain the answer, refuse" |
| **No source citations** | User can't verify the answer | Always include the source URL or page title |
| **Skipping retrieval testing** | You assume retrieval works but it doesn't | Test retrieval independently before connecting the LLM |
| **Crawling too aggressively** | Fetching 1000 pages when 20 would suffice; getting rate-limited | Set MAX_PAGES, add delays between requests |
| **No fallback for empty retrieval** | If retriever returns nothing, LLM gets no context | Detect empty results and return a clear "no information" response |

## 24. Production Improvement Ideas

1. **Scheduled re-crawl** — Websites change; re-crawl weekly or on webhook
2. **Multi-site ingestion** — Crawl docs site, blog, and help center into one index
3. **Hybrid search** — Combine vector similarity with BM25 keyword search
4. **Conversation memory** — Track multi-turn conversations for follow-up questions
5. **Answer confidence scoring** — Show users how confident the chatbot is
6. **Feedback loop** — Let users rate answers; use bad answers to improve prompts
7. **Widget deployment** — Embed as a chat widget on the website itself
8. **Structured data extraction** — Parse pricing tables and feature matrices into structured fields
9. **Persistent vector store** — Use a Chroma persistent directory or Pinecone for production

## 25. Exercises

### Exercise 1: Crawl Your Own Website

In [ ]:
# ── Exercise 1: Change the URL and run the pipeline ───
# Uncomment and modify:

# MY_URL = "https://your-website-here.com"
# my_pages = crawl_website(MY_URL, max_pages=10)
#
# if my_pages:
#     my_docs = []
#     for page in my_pages:
#         chunks = splitter.split_text(page["text"])
#         for j, chunk in enumerate(chunks):
#             my_docs.append(Document(
#                 page_content=chunk,
#                 metadata={"source": page["url"], "title": page["title"], "chunk_index": j},
#             ))
#     my_vs = Chroma.from_documents(my_docs, embeddings, collection_name="my_site")
#     # Now query with your own questions!

print("Exercise 1: Replace MY_URL with your website and uncomment the code above.")
print("Then ask questions about YOUR website content!")

### Exercise 2: Improve the Extraction — Handling Tables & Lists

In [ ]:
# ── Exercise 2: Better HTML extraction ────────────────
# The current extractor loses table structure. Try improving it.

def extract_text_v2(html: str, url: str) -> dict:
    """
    Enhanced extraction that preserves table and list structure.
    """
    soup = BeautifulSoup(html, "html.parser")
    title = soup.title.string.strip() if soup.title and soup.title.string else ""

    for tag in soup.find_all(["script", "style", "nav", "footer", "header",
                              "aside", "noscript", "iframe"]):
        tag.decompose()

    # Convert tables to structured text
    for table in soup.find_all("table"):
        rows = []
        for tr in table.find_all("tr"):
            cells = [td.get_text(strip=True) for td in tr.find_all(["td", "th"])]
            if cells:
                rows.append(" | ".join(cells))
        table.replace_with(soup.new_string("\n".join(rows)))

    # Convert lists to bullet points
    for ul in soup.find_all(["ul", "ol"]):
        items = []
        for li in ul.find_all("li", recursive=False):
            items.append(f"• {li.get_text(strip=True)}")
        ul.replace_with(soup.new_string("\n".join(items)))

    content_area = soup.find("main") or soup.find("article") or soup.body or soup
    text = content_area.get_text(separator="\n", strip=True)
    lines = [line.strip() for line in text.split("\n") if line.strip()]

    return {"url": url, "title": title, "text": "\n".join(lines)}


# Test on a sample HTML with a table
test_html = """
<html><head><title>Test</title></head><body>
<main>
<h1>Pricing</h1>
<table><tr><th>Plan</th><th>Price</th></tr>
<tr><td>Starter</td><td>$9/mo</td></tr>
<tr><td>Pro</td><td>$29/mo</td></tr></table>
<ul><li>Feature A</li><li>Feature B</li><li>Feature C</li></ul>
</main></body></html>
"""

v1 = extract_text(test_html, "test")
v2 = extract_text_v2(test_html, "test")
print("v1 (basic):")
print(v1["text"])
print("\nv2 (enhanced):")
print(v2["text"])
print("\n→ v2 preserves table structure and list formatting!")

### Exercise 3: Multi-turn Conversation

In [ ]:
# ── Exercise 3: Add conversation memory ───────────────
# The current chatbot is stateless — each question is independent.
# Add conversation history so follow-up questions work.

CONVERSATIONAL_SYSTEM = """You are a helpful FAQ assistant. Answer using ONLY the provided context.
Use the conversation history to understand follow-up questions.
If the context doesn't contain the answer, say so."""

CONVERSATIONAL_USER = """CONTEXT:
{context}

CONVERSATION HISTORY:
{history}

NEW QUESTION: {question}

Answer:"""


def ask_with_history(question: str, history: list[dict]) -> dict:
    """RAG with conversation memory."""
    docs = retriever.invoke(question)
    context = format_context(docs)

    history_text = "\n".join(
        f"User: {h['question']}\nAssistant: {h['answer']}" for h in history[-3:]
    ) if history else "(none)"

    resp = llm.invoke([
        SystemMessage(content=CONVERSATIONAL_SYSTEM),
        HumanMessage(content=CONVERSATIONAL_USER.format(
            context=context, history=history_text, question=question,
        )),
    ])
    answer = clean(resp.content)
    return {"question": question, "answer": answer}


# Simulate multi-turn conversation
print("MULTI-TURN CONVERSATION")
print("=" * 60)

conversation = []
turns = [
    "What plans do you offer?",
    "Which one includes API access?",       # Follow-up — "which one" refers to plans
    "And what are the rate limits for that?",  # Follow-up — "that" refers to the plan with API
]

for q in turns:
    result = ask_with_history(q, conversation)
    conversation.append(result)
    print(f"User: {q}")
    print(f"Bot:  {result['answer']}")
    print()

### Mini Challenge

1. **Semantic cache:** Store (question hash → answer) pairs. If a new question is semantically similar (cosine similarity > 0.95) to a cached question, return the cached answer without calling the LLM. How much does this speed things up?

2. **Hybrid retrieval:** Combine ChromaDB vector search with keyword matching (BM25). Does precision improve on exact-term queries like "HIPAA" or "99.99% SLA"?

3. **Auto-FAQ generation:** Instead of answering user questions, generate a list of 10 common FAQs from the website content itself. Use a prompt like "Based on this website text, what are the 10 most likely questions a visitor would ask?"

4. **Chunk deduplication:** Some chunks overlap significantly. Before indexing, compute pairwise similarity and remove near-duplicate chunks. Does this improve or hurt retrieval?

## 26. Session Summary

In [ ]:
print("SESSION SUMMARY")
print("=" * 50)

print(f"\nData source: {data_source}")
print(f"Pages crawled: {len(pages)}")
print(f"Total text: {sum(len(p['text']) for p in pages):,} chars")
print(f"Chunks indexed: {vectorstore._collection.count()}")

print(f"\nQuestions answered: {len(all_results)}")
print(f"Grounding test: {grounding_pass}/{len(GROUNDING_TESTS)} correctly refused")

if eval_results:
    avg_f = sum(e.get('faithfulness', 0) for e in eval_results) / len(eval_results)
    avg_r = sum(e.get('relevance', 0) for e in eval_results) / len(eval_results)
    print(f"\nAverage Faithfulness: {avg_f:.1f}/5")
    print(f"Average Relevance:   {avg_r:.1f}/5")

print(f"\nConfiguration:")
print(f"  LLM: {LLM_MODEL}")
print(f"  Embeddings: {EMBED_MODEL}")
print(f"  Chunk: {CHUNK_SIZE} chars, {CHUNK_OVERLAP} overlap")
print(f"  Retrieval: top-{TOP_K}")

## 27. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Clean HTML aggressively** — nav, footer, scripts, sidebars all pollute embeddings and waste tokens |
| 2 | **Chunk size is a tradeoff** — smaller (300-500 chars) for precision, larger (800-1000) for context |
| 3 | **Test retrieval before generation** — if retrieval is bad, no amount of prompt engineering fixes the answer |
| 4 | **Grounding is non-negotiable** — the system prompt must force refusal when context is insufficient |
| 5 | **Citations build trust** — always show users where the answer came from |
| 6 | **Naive vs RAG comparison** proves the value — LLMs without context hallucinate website-specific details |
| 7 | **Evaluation needs both dimensions** — faithfulness (is it supported?) and relevance (does it answer the question?) |
| 8 | **Edge cases reveal gaps** — comparison questions, negation, and multi-page answers stress-test the pipeline |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*